# CNN Training Pipeline (MRI images)

This notebook trains a simple CNN on the MRI image folders stored under `data/raw/MRI/Alzhiemer/combined_images` (one subfolder per class). It includes data loading, a lightweight CNN model, training, evaluation, and checkpoint saving.

In [1]:
import os
from pathlib import Path
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

print('Imports ready')

Imports ready


In [2]:
# Configuration
SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3
IMG_SIZE = 128
# Paths: allow override via env var
# Prefer an absolute path based on the repository root (search upwards for a folder containing 'data')
cwd = Path('.').resolve()
REPO_ROOT = cwd
for p in [cwd] + list(cwd.parents):
    if (p / 'data').exists():
        REPO_ROOT = p
        break
DEFAULT_ROOT = REPO_ROOT / 'data' / 'raw' / 'MRI' / 'Alzhiemer' / 'combined_images'
DATA_ROOT = Path(os.environ.get('ALZHEIMER_MRI_ROOT', str(DEFAULT_ROOT)))
# If default path doesn't exist, try to auto-discover likely image folders under data/
if not DATA_ROOT.exists():
    candidates = list(Path('data').rglob('combined_images')) if Path('data').exists() else []
    # keep only directories that look like class-folder datasets (contain subdirs or image files)
    filtered = []
    for c in candidates:
        try:
            entries = list(c.iterdir())
        except Exception:
            entries = []
        if any(e.is_dir() for e in entries) or any(e.suffix.lower() in ['.png', '.jpg', '.jpeg'] for e in entries):
            filtered.append(c)
    if filtered:
        DATA_ROOT = filtered[0]
        print(f'Using discovered data root: {DATA_ROOT}')
    else:
        raise FileNotFoundError(f'Data root not found: {DATA_ROOT}. Set ALZHEIMER_MRI_ROOT to your images folder or place images under data/raw/MRI/Alzhiemer/combined_images')
CHECKPOINT_PATH = Path('models/saved/cnn_best.pth')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Device: {DEVICE}')
print(f'Data root: {DATA_ROOT}')

Device: cpu
Data root: C:\ALZHEIMER_DETECTION\data\raw\MRI\Alzhiemer\combined_images


In [3]:
# Simple image dataset discovery and split
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Data root not found: {DATA_ROOT}')

# Use torchvision.datasets.ImageFolder for folder-structured datasets
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

full_dataset = datasets.ImageFolder(root=str(DATA_ROOT), transform=transform)
class_names = full_dataset.classes
print('Discovered classes:', class_names)

# Create train/val/test splits indices
indices = list(range(len(full_dataset)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=[full_dataset.targets[i] for i in indices], random_state=SEED)
train_idx, val_idx = train_test_split(train_idx, test_size=0.125, stratify=[full_dataset.targets[i] for i in train_idx], random_state=SEED)  # 0.125 of 0.8 -> 0.1 overall validation

from torch.utils.data import Subset
train_ds = Subset(full_dataset, train_idx)
val_ds = Subset(full_dataset, val_idx)
test_ds = Subset(full_dataset, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

Discovered classes: ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
Train: 30800, Val: 4400, Test: 8800


In [4]:
# Define a small CNN (for demonstration)
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*4*4, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=len(class_names)).to(DEVICE)
print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): AdaptiveAvgPool2d(output_size=(4, 4))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1024, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=4, bias=True)
  )
)


In [5]:
# Training utilities and loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())
        all_targets.extend(yb.detach().cpu().numpy().tolist())
    avg_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    return avg_loss, acc

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            running_loss += loss.item() * xb.size(0)
            all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())
            all_targets.extend(yb.detach().cpu().numpy().tolist())
    avg_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    return avg_loss, acc, all_targets, all_preds

# Training loop
best_val = -1.0
for epoch in range(1, NUM_EPOCHS+1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    print(f'Epoch {epoch}/{NUM_EPOCHS}  Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}')
    if val_acc > best_val:
        best_val = val_acc
        CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
        torch.save({'model_state_dict': model.state_dict(), 'class_names': class_names}, CHECKPOINT_PATH)
        print(f'Saved best model to {CHECKPOINT_PATH} (Val Acc: {best_val:.4f})')

# Final evaluation on test set
if CHECKPOINT_PATH.exists():
    ck = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ck['model_state_dict'])
    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, DEVICE)
    print(f'Test Loss: {test_loss:.4f}  Test Acc: {test_acc:.4f}')
    print('')
    print('Classification report:')
    from sklearn.metrics import classification_report
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
else:
    print(f'No checkpoint found at {CHECKPOINT_PATH}, skipping final evaluation.')

Epoch 1/15  Train Loss: 1.1224  Train Acc: 0.4655  Val Loss: 0.7881  Val Acc: 0.6239
Saved best model to models\saved\cnn_best.pth (Val Acc: 0.6239)
Epoch 2/15  Train Loss: 0.7770  Train Acc: 0.6321  Val Loss: 0.6914  Val Acc: 0.6650
Saved best model to models\saved\cnn_best.pth (Val Acc: 0.6650)
Epoch 3/15  Train Loss: 0.6932  Train Acc: 0.6765  Val Loss: 0.6278  Val Acc: 0.7061
Saved best model to models\saved\cnn_best.pth (Val Acc: 0.7061)
Epoch 4/15  Train Loss: 0.6284  Train Acc: 0.7077  Val Loss: 0.5767  Val Acc: 0.7395
Saved best model to models\saved\cnn_best.pth (Val Acc: 0.7395)
Epoch 5/15  Train Loss: 0.5677  Train Acc: 0.7439  Val Loss: 0.5311  Val Acc: 0.7639
Saved best model to models\saved\cnn_best.pth (Val Acc: 0.7639)
Epoch 6/15  Train Loss: 0.5059  Train Acc: 0.7709  Val Loss: 0.4372  Val Acc: 0.8125
Saved best model to models\saved\cnn_best.pth (Val Acc: 0.8125)
Epoch 7/15  Train Loss: 0.4485  Train Acc: 0.8010  Val Loss: 0.4086  Val Acc: 0.8198
Saved best model to m